# Symposia Getting Started

This notebook shows the smallest useful Symposia flow:

1. pick a claim
2. run `validate(...)`
3. inspect the public result fields

Before you run it, set at least one of these environment variables:
- `OPENAI_API_KEY`
- `ANTHROPIC_API_KEY`
- `GOOGLE_API_KEY`

Run the cells from top to bottom.

In [46]:
import os
import sys
import time
from pathlib import Path
from pprint import pprint

# Jupyter already has an event loop. This keeps live mode working in notebooks.
try:
    import nest_asyncio
except ImportError:
    nest_asyncio = None
else:
    nest_asyncio.apply()

ROOT = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()
sys.path.insert(0, str(ROOT))

from symposia import validate

CLAIM = (
    "Vitamin C deficiency causes scurvy. "
    "Scurvy is a serious disease that can lead to bleeding gums and poor wound healing. "
    "Vitamin C supplementation prevents scurvy."
)

In [47]:
def show_result(result):
    print("verdict   :", result.verdict)
    print("agreement :", result.agreement)
    print("caveats   :", result.caveats or ["none"])
    print("trace     :", type(result.trace).__name__)

## Run One Review

This cell makes one live API call and prints the public result fields.

In [48]:
providers = []
if os.getenv("OPENAI_API_KEY"):
    providers.append("openai")
if os.getenv("ANTHROPIC_API_KEY"):
    providers.append("anthropic")
if os.getenv("GOOGLE_API_KEY"):
    providers.append("google")

if not providers:
    print("No provider keys found.")
    print("Set OPENAI_API_KEY, ANTHROPIC_API_KEY, or GOOGLE_API_KEY, then rerun this cell.")
else:
    print("Using provider keys for:", providers)

    t0 = time.perf_counter()
    live_result = validate(
        content=CLAIM,
        domain="general",
        live=True,
    )
    elapsed_ms = round((time.perf_counter() - t0) * 1000, 1)

    show_result(live_result)
    pprint(f"elapsed_ms:{elapsed_ms}" )

Using provider keys for: ['openai', 'anthropic', 'google']
verdict   : validated
agreement : 1.0
caveats   : ['none']
trace     : AdjudicationTrace
'elapsed_ms:998.3'


In [49]:
pprint(live_result.trace.to_canonical_json(), indent=2, compact=True)

('{"core_trace":{"aggregation_outcome":[{"contradiction_score":0.0,"subclaim_id":"sc_001","sufficiency_score":1.0,"support_score":1.0}],"juror_votes":[{"confidence":0.9,"contradicted":false,"error_code":null,"juror_id":"juror_1","parsed_ok":true,"provider_model":"gpt-4o-mini","subclaim_id":"sc_001","sufficient":true,"supported":true}],"profile_set_selected":"general_default_v1","run_id":"run_5f054b4296aa","subclaims":[{"subclaim_id":"sc_001","text":"Vitamin '
 'C deficiency causes scurvy. Scurvy is a serious disease that can lead to '
 'bleeding gums and poor wound healing. Vitamin C supplementation prevents '
 'scurvy."}]},"events":[{"entity_id":"run_5f054b4296aa","event_index":1,"event_type":"run_started","message":"Round '
 '0 run '
 'started.","metadata":{"juror_mode":"llm","subclaim_count":1}},{"entity_id":"run_5f054b4296aa","event_index":2,"event_type":"run_policy_applied","message":"Run-level '
 'execution policy '
 'applied.","metadata":{"decomposition_mode":"holistic","juror_m